# 🔬 Phase 4 — Controlled Class Imbalance: Sweep & Benchmark Data Generation

**Project**: Low-Resource Language (Ukrainian) Sentiment & Emotion Analysis  
**Dataset**: [KSE-RESEARCH-Group/UAReviews](https://huggingface.co/datasets/KSE-RESEARCH-Group/UAReviews) (11,580 Ukrainian reviews)  
**Embeddings**: Pre-encoded 1024-d dense vectors from [Qwen/Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B)  
**Hardware**: Torch-CUDA GPU Runtime  

### Experimental Focus:
This notebook handles the **experimental sweep execution** and **benchmark dataset generation**:
1. **Oversampling Operators**: Pure NumPy implementations of `duplication`, classic `smote`, norm-preserving `smote_renorm`, and retrieval-based `embsmote` (Inoshita, 2026).
2. **Controlled Sweep**: Imbalance ratios $\rho \in \{0.10, 0.25, 0.50, 0.75, 1.00\}$ across `LinearSVC` and `TorchMLPClassifier`.
3. **Dual Evaluation**: In-distribution `test` set (1,737) and out-of-distribution `challenge` set (1,737).
4. **Resumable Disk Checkpointing**: Every condition is saved as an independent JSON file in `cache/phase4/`.
5. **Benchmark Summary**: Results are aggregated into `cache/phase4/phase4_sweep_summary.csv`.

> **Note**: All plotting, gain curves, robustness trade-off analysis, and publication figures have been separated into **`phase_4_analysis.ipynb`**.


In [ ]:
# 1. Setup & GPU Verification
!pip install -q scikit-learn matplotlib seaborn torch pandas numpy tqdm

import os
import gc
import json
import time
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.svm import LinearSVC
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Runtime: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# 2. Storage Setup & Load Phase 1-3 Artifacts
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/lrl_sentiment_short'
except Exception:
    BASE_DIR = os.path.abspath('./lrl_cache')

CACHE_DIR = os.path.join(BASE_DIR, 'cache')
EMBED_DIR = os.path.join(CACHE_DIR, 'embeddings')
PHASE4_DIR = os.path.join(CACHE_DIR, 'phase4')
os.makedirs(PHASE4_DIR, exist_ok=True)

with open(os.path.join(CACHE_DIR, 'label_encoder.json'), 'r', encoding='utf-8') as f:
    label_map = json.load(f)
emotion2id = label_map['emotion2id']
id2emotion = {int(k): v for k, v in label_map['id2emotion'].items()}
EMOTION_LABELS = [id2emotion[i] for i in range(len(id2emotion))]
NUM_CLASSES = len(EMOTION_LABELS)

with open(os.path.join(CACHE_DIR, 'split_indices.json'), 'r') as f:
    split_indices = json.load(f)

train_idx = np.array(split_indices['train'])
test_idx = np.array(split_indices['test'])
chal_idx = np.array(split_indices['challenge'])

X_all = np.load(os.path.join(EMBED_DIR, 'qwen3_embeddings_all.npy'))
df = pd.read_parquet(os.path.join(CACHE_DIR, 'ua_reviews_clean.parquet'))
y_all = df['label'].values

X_train, y_train = X_all[train_idx], y_all[train_idx]
X_test, y_test = X_all[test_idx], y_all[test_idx]
X_chal, y_chal = X_all[chal_idx], y_all[chal_idx]

print(f"Loaded artifacts:")
print(f"  X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"  X_test:  {X_test.shape}  | y_test:  {y_test.shape}")
print(f"  X_chal:  {X_chal.shape}  | y_chal:  {y_chal.shape}")
print(f"  Classes: {EMOTION_LABELS}")


## 3. Oversampling Strategies Implementation

We implement 4 distinct data augmentation mechanisms from scratch in pure NumPy/Scikit-learn, operating on normalized 1024-d embeddings:
1. **`duplication`** (Random Oversampling): Sample minority instances uniformly at random with replacement.
2. **`smote`** (Classic Embedding SMOTE): Intra-class linear interpolation along the segment joining a minority instance to a random $k$-nearest neighbor:
   $$z_{\text{synth}} = z_i + \lambda(z_{nn} - z_i), \quad \lambda \sim U(0, 1)$$
3. **`smote_renorm`** (L2-Renormalized SMOTE): Synthetic vectors from SMOTE are re-projected onto the unit hypersphere:
   $$z_{\text{renorm}} = \frac{z_{\text{synth}}}{\|z_{\text{synth}}\|_2}$$
   This counteracts norm-shrinkage inherent in linear interpolation of high-dimensional directional vectors.
4. **`embsmote`** (Retrieval-Based EmbSMOTE; Inoshita 2026): Intra-class interpolation is computed, followed by nearest-neighbor retrieval from the real training set points of that class:
   $$x^* = \arg\min_{x \in \mathcal{D}_{\text{train}}^{(c)}} \|z_{\text{synth}} - x\|$$
   This structurally guarantees that every synthetic sample is an authentic member of the class manifold.


In [ ]:
# 3. Oversampling Operators
SMOTE_K = 5

def oversample_duplication(X_c, n_synth, rng):
    # Random oversampling with replacement
    if n_synth <= 0 or len(X_c) == 0:
        return np.empty((0, X_c.shape[1]), dtype=np.float32)
    indices = rng.choice(len(X_c), size=n_synth, replace=True)
    return X_c[indices].copy()

def oversample_smote(X_c, n_synth, k=SMOTE_K, rng=None, renorm=False):
    # Classic SMOTE interpolation in embedding space
    if n_synth <= 0 or len(X_c) == 0:
        return np.empty((0, X_c.shape[1]), dtype=np.float32)
    
    n_samples, d = X_c.shape
    k_eff = max(1, min(k, n_samples - 1))
    
    if n_samples <= 1:
        return oversample_duplication(X_c, n_synth, rng)
    
    nbrs = NearestNeighbors(n_neighbors=k_eff + 1, metric='cosine').fit(X_c)
    _, indices = nbrs.kneighbors(X_c)
    
    base_idx = rng.choice(n_samples, size=n_synth, replace=True)
    neighbor_col = rng.integers(1, k_eff + 1, size=n_synth)
    chosen_nn_idx = indices[base_idx, neighbor_col]
    
    lambdas = rng.uniform(0.0, 1.0, size=(n_synth, 1)).astype(np.float32)
    diff = X_c[chosen_nn_idx] - X_c[base_idx]
    synth = X_c[base_idx] + lambdas * diff
    
    if renorm:
        norms = np.linalg.norm(synth, axis=1, keepdims=True)
        norms = np.maximum(norms, 1e-8)
        synth = synth / norms
        
    return synth.astype(np.float32)

def oversample_embsmote(X_c, n_synth, k=SMOTE_K, rng=None):
    # EmbSMOTE (Inoshita, 2026): Intra-class interpolation + nearest-neighbor retrieval
    if n_synth <= 0 or len(X_c) == 0:
        return np.empty((0, X_c.shape[1]), dtype=np.float32)
    
    # 1. Generate interpolated candidate vectors
    candidates = oversample_smote(X_c, n_synth, k=k, rng=rng, renorm=True)
    
    # 2. Retrieve nearest real instance in X_c for each candidate
    nbrs = NearestNeighbors(n_neighbors=1, metric='cosine').fit(X_c)
    _, ret_indices = nbrs.kneighbors(candidates)
    
    retrieved = X_c[ret_indices.squeeze(1)].copy()
    return retrieved.astype(np.float32)

def build_oversampled_dataset(X_tr, y_tr, strategy, rho, rng=None):
    # Builds augmented training dataset where minority classes are oversampled 
    # to reach target size: target_size(c) = max(N_c, round(rho * N_majority)).
    if rng is None:
        rng = np.random.default_rng(SEED)
        
    class_counts = {c: int((y_tr == c).sum()) for c in range(NUM_CLASSES)}
    maj_count = max(class_counts.values())
    
    X_parts = []
    y_parts = []
    added_stats = {}
    
    for c in range(NUM_CLASSES):
        X_c = X_tr[y_tr == c]
        n_orig = len(X_c)
        X_parts.append(X_c)
        y_parts.append(np.full(n_orig, c, dtype=np.int64))
        
        if strategy == "none":
            added_stats[c] = 0
            continue
            
        target_count = max(n_orig, int(round(rho * maj_count)))
        n_synth = target_count - n_orig
        
        if n_synth > 0:
            if strategy == "duplication":
                X_synth = oversample_duplication(X_c, n_synth, rng)
            elif strategy == "smote":
                X_synth = oversample_smote(X_c, n_synth, k=SMOTE_K, rng=rng, renorm=False)
            elif strategy == "smote_renorm":
                X_synth = oversample_smote(X_c, n_synth, k=SMOTE_K, rng=rng, renorm=True)
            elif strategy == "embsmote":
                X_synth = oversample_embsmote(X_c, n_synth, k=SMOTE_K, rng=rng)
            else:
                raise ValueError(f"Unknown strategy: {strategy}")
                
            X_parts.append(X_synth)
            y_parts.append(np.full(n_synth, c, dtype=np.int64))
            added_stats[c] = int(n_synth)
        else:
            added_stats[c] = 0
            
    X_aug = np.vstack(X_parts).astype(np.float32)
    y_aug = np.concatenate(y_parts).astype(np.int64)
    
    perm = rng.permutation(len(y_aug))
    return X_aug[perm], y_aug[perm], added_stats


## 4. Classifiers Adapted for Torch-CUDA

We evaluate two models:
1. **`LinearSVC`**: Linear Support Vector Classifier with One-vs-Rest multi-class strategy ($C=1.0$).
2. **`TorchMLPClassifier`**: Fully GPU-accelerated multi-layer perceptron (Linear -> BatchNorm -> ReLU -> Dropout -> Linear -> BatchNorm -> ReLU -> Dropout -> Linear), trained with Cross-Entropy Loss, AdamW optimizer, and early stopping on a stratified validation set.


In [ ]:
# 4. Classifier Definitions
import torch.nn as nn

class TorchMLP(nn.Module):
    def __init__(self, in_dim=1024, hidden_dims=(256, 64), num_classes=NUM_CLASSES, dropout=0.2):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(d, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            d = h
        layers.append(nn.Linear(d, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.net(x)

class TorchMLPClassifier:
    # sklearn-compatible wrapper for GPU-accelerated PyTorch MLP
    def __init__(self, hidden_dims=(256, 64), lr=1e-3, weight_decay=1e-4, batch_size=128,
                 epochs=60, patience=8, random_state=SEED):
        self.hidden_dims = hidden_dims
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.epochs = epochs
        self.patience = patience
        self.random_state = random_state
        self.model = None

    def fit(self, X, y, X_val=None, y_val=None):
        torch.manual_seed(self.random_state)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.int64)
        
        if X_val is None or y_val is None:
            from sklearn.model_selection import train_test_split
            X_tr, X_va, y_tr, y_va = train_test_split(
                X, y, test_size=0.10, random_state=self.random_state, stratify=y
            )
        else:
            X_tr, y_tr, X_va, y_va = X, y, X_val, y_val

        ds_tr = TensorDataset(torch.as_tensor(X_tr, dtype=torch.float32), torch.as_tensor(y_tr, dtype=torch.long))
        loader = DataLoader(ds_tr, batch_size=self.batch_size, shuffle=True)
        
        X_va_t = torch.as_tensor(X_va, dtype=torch.float32, device=DEVICE)
        y_va_t = torch.as_tensor(y_va, dtype=torch.long, device=DEVICE)

        self.model = TorchMLP(in_dim=X.shape[1], hidden_dims=self.hidden_dims, num_classes=NUM_CLASSES).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.CrossEntropyLoss()

        best_score, best_state, wait = -1.0, None, 0
        for epoch in range(1, self.epochs + 1):
            self.model.train()
            for bx, by in loader:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                opt.zero_grad(set_to_none=True)
                loss = loss_fn(self.model(bx), by)
                loss.backward()
                opt.step()

            self.model.eval()
            with torch.no_grad():
                va_preds = self.model(X_va_t).argmax(dim=-1).cpu().numpy()
                va_f1 = f1_score(y_va, va_preds, average='macro', zero_division=0)

            if va_f1 > best_score + 1e-4:
                best_score = va_f1
                wait = 0
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
            else:
                wait += 1
                if wait >= self.patience:
                    break

        if best_state is not None:
            self.model.load_state_dict(best_state)
            self.model.to(DEVICE)
        return self

    @torch.no_grad()
    def predict(self, X):
        self.model.eval()
        X_t = torch.as_tensor(X, dtype=torch.float32)
        ds = TensorDataset(X_t)
        loader = DataLoader(ds, batch_size=256, shuffle=False)
        preds = []
        for (bx,) in loader:
            p = self.model(bx.to(DEVICE)).argmax(dim=-1).cpu().numpy()
            preds.append(p)
        return np.concatenate(preds)


## 5. Sweep Driver with Resumable Disk Checkpointing

We sweep across:
- **Strategies**: `['none', 'duplication', 'smote', 'smote_renorm', 'embsmote']`
- **Target Imbalance Ratios ($\rho$)**: `[0.10, 0.25, 0.50, 0.75, 1.00]`
- **Classifiers**: `['linearsvc', 'mlp']`

Every condition result is cached in `cache/phase4/{cond_id}.json`. If execution is paused or interrupted, reruns skip existing conditions instantly.


In [ ]:
# 5. Sweep Driver Implementation
STRATEGIES = ["none", "duplication", "smote", "smote_renorm", "embsmote"]
RHOS = [0.10, 0.25, 0.50, 0.75, 1.00]
CLASSIFIERS = ["linearsvc", "mlp"]

def score_predictions(y_true, y_pred):
    return {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "per_class_f1": {EMOTION_LABELS[i]: float(f) for i, f in enumerate(f1_score(y_true, y_pred, average=None, zero_division=0))},
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES))).tolist()
    }

def run_condition(strategy, rho, clf_name, seed=SEED):
    cond_id = f"strat_{strategy}_rho{rho}__{clf_name}"
    out_path = os.path.join(PHASE4_DIR, f"{cond_id}.json")
    
    if os.path.exists(out_path):
        with open(out_path, 'r') as f:
            cached = json.load(f)
        print(f"[cached] {cond_id} -> Test Macro F1: {cached['test_metrics']['macro_f1']:.4f}")
        return cached

    t0 = time.time()
    rng = np.random.default_rng(seed)
    
    if strategy == "none":
        X_train_c, y_train_c = X_train.copy(), y_train.copy()
        added_stats = {c: 0 for c in range(NUM_CLASSES)}
    else:
        X_train_c, y_train_c, added_stats = build_oversampled_dataset(
            X_train, y_train, strategy=strategy, rho=rho, rng=rng
        )
        
    n_synth_total = sum(added_stats.values())
    print(f"[{cond_id}] Train size: {len(X_train_c)} (Added synthetics: {n_synth_total})")

    if clf_name == "linearsvc":
        model = LinearSVC(C=1.0, max_iter=5000, random_state=seed)
        model.fit(X_train_c, y_train_c)
    elif clf_name == "mlp":
        model = TorchMLPClassifier(random_state=seed)
        model.fit(X_train_c, y_train_c)
    else:
        raise ValueError(clf_name)

    test_preds = model.predict(X_test)
    chal_preds = model.predict(X_chal)
    
    test_metrics = score_predictions(y_test, test_preds)
    chal_metrics = score_predictions(y_chal, chal_preds)
    elapsed = time.time() - t0

    result = {
        "strategy": strategy,
        "rho": rho,
        "clf": clf_name,
        "train_size": int(len(X_train_c)),
        "n_synthetics": int(n_synth_total),
        "synthetics_per_class": added_stats,
        "elapsed_sec": float(elapsed),
        "test_metrics": test_metrics,
        "challenge_metrics": chal_metrics
    }
    
    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2)
        
    print(f"[done] {cond_id} | Test Macro F1: {test_metrics['macro_f1']:.4f} | "
          f"Chal Macro F1: {chal_metrics['macro_f1']:.4f} | Time: {elapsed:.1f}s")
    
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
        
    return result


In [ ]:
# 6. Execute Experimental Sweep
all_results = []

# Baseline conditions (none) at rho=1.0
for clf in CLASSIFIERS:
    all_results.append(run_condition("none", 1.00, clf))

# Sweep oversampling strategies across rhos
for rho in RHOS:
    for strategy in ["duplication", "smote", "smote_renorm", "embsmote"]:
        for clf in CLASSIFIERS:
            all_results.append(run_condition(strategy, rho, clf))

print(f"\nSweep Complete! {len(all_results)} total conditions evaluated.")


## 6. Benchmark Data Generation & Summary CSV Compilation

Consolidates all cached JSON condition checkpoints into a single structured benchmark dataset `phase4_sweep_summary.csv`.


In [ ]:
# 7. Aggregate Results into DataFrame
rows = []
for r in all_results:
    strat = r['strategy']
    rho = r['rho']
    clf = r['clf']
    tm = r['test_metrics']
    cm = r['challenge_metrics']
    
    row = {
        'Strategy': strat,
        'Rho': rho,
        'Classifier': 'LinearSVC' if clf == 'linearsvc' else 'TorchMLP',
        'Train Size': r['train_size'],
        'Test Macro F1': tm['macro_f1'],
        'Test Weighted F1': tm['weighted_f1'],
        'Test Acc': tm['accuracy'],
        'Chal Macro F1': cm['macro_f1'],
        'Chal Weighted F1': cm['weighted_f1'],
        'Fear F1': tm['per_class_f1']['Fear'],
        'Surprise F1': tm['per_class_f1']['Surprise'],
        'Disgust F1': tm['per_class_f1']['Disgust'],
        'Sadness F1': tm['per_class_f1']['Sadness'],
        'Elapsed (s)': r['elapsed_sec']
    }
    rows.append(row)

results_df = pd.DataFrame(rows)
summary_csv = os.path.join(PHASE4_DIR, 'phase4_sweep_summary.csv')
results_df.to_csv(summary_csv, index=False)
print(f"Summary saved: {summary_csv}")
print(results_df.head(10))


## ✅ Benchmark Data Generation Complete!

All 42 experimental conditions have been evaluated and checkpointed to disk:
- **Condition Checkpoints**: `cache/phase4/strat_{strategy}_rho{rho}__{clf}.json`
- **Consolidated Benchmark CSV**: `cache/phase4/phase4_sweep_summary.csv`

---

### ➡️ Next Step: Open `phase_4_analysis.ipynb`
All empirical analysis, gain curves (test and challenge vs baseline), robustness trade-offs, and publication figures (Figures 1–7) are located in the companion analysis notebook:  
**[`phase_4_analysis.ipynb`](./phase_4_analysis.ipynb)**.

Open `phase_4_analysis.ipynb` to generate all charts instantly without re-running the GPU sweep.
